# HeinzleNet: Instantiating and Inspecting the Inversion Model

`notebooks/heinzlenet.md` already explains **why** HeinzleNet is built the way it is (layer mixing,
spatial encoding, temporal TCN, FiLM decoder, and why each stage exists). This notebook is the hands-on
companion: we actually **instantiate** every building block from `src/mich/models/blocks.py`, run real
tensors through them, inspect shapes at each stage, and show exactly how `config/model/*.yaml` maps onto
each constructor.

Everything below uses real repo classes with no reimplementation.

In [ ]:
import json

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

torch.manual_seed(0)
%matplotlib inline

## 0. A real BOLD sample to work with

We'll use a few samples from `data/simulations-three_layer_single_source_single_active.h5`, a 3-layer,
10×10-grid dataset generated exactly as in `01_running_simulations.ipynb`. Its layer count (`L=3`) and
grid size (`10×10`) happen to match `config/model/default.yaml`, so we can later run the actual default
model config on real data without changing any dimensions.

In [ ]:
DATA_PATH = "../data/simulations-three_layer_single_source_single_active.h5"  # or wherever your data is stored
N_SAMPLES = 4

with h5py.File(DATA_PATH, "r") as f:
    layer_keys = sorted(k for k in f.keys() if k.startswith("layer"))
    bold = np.stack(
        [f[layer]["bold"][:N_SAMPLES] for layer in layer_keys], axis=1
    )  # (N, L, T, H, W)
    x_true = np.stack([f[layer]["x"][:N_SAMPLES] for layer in layer_keys], axis=1)
    source_layer = f["meta"]["sources"]["layer"][:N_SAMPLES]
    source_position = f["meta"]["sources"]["position"][:N_SAMPLES]

print("bold:", bold.shape, bold.dtype)
N, L, T, H, W = bold.shape
bold_t = torch.from_numpy(bold.astype(np.float32))
print("as a float32 tensor:", bold_t.shape)

## 1. The building blocks, one at a time

Before assembling the real (32/64-channel) model, we instantiate each block from `blocks.py` at tiny,
easy-to-read dimensions, the same style used in `tests/test_blocks.py`, purely to see how the shape of
the tensor changes at each stage. `L` here matches our loaded data (3 layers).

### 1a. `MaskedLayerMixing`: let layers talk to their immediate predecessor

Input `[B, L, T, H, W]` maps to output `[B, T, L, C, H, W]`. A 1×1 conv over the layer axis, masked so
layer `i` can only receive from itself and layer `i-1` (deep to superficial), followed by a per-layer
channel expansion to `C` channels.

In [ ]:
from mich.models.blocks import MaskedLayerMixing

layer_mix = MaskedLayerMixing(L=3, C=4, init_identity=True)
print("layer-mixing mask (row=target layer, col=source layer):")
print(layer_mix.mask.squeeze().int().numpy())

small_x = bold_t[:2]  # [B=2, L=3, T, H, W]
mixed = layer_mix(small_x)
print("input :", small_x.shape, " -> output:", mixed.shape, " ([B, T, L, C, H, W])")

Each layer receives from itself (the diagonal) and the layer directly below it (the sub-diagonal). The
superficial layer (index 2) is influenced by the middle layer (index 1), which is influenced by the deep
layer (index 0), but not vice versa. This mirrors the drainage direction in the forward Balloon model
(`CortexLayer.drain_from` in `balloon.py`, used in `01_running_simulations.ipynb`).

### 1b. `SpatialEncoder`: local spatial features via depthwise-separable convs

In [ ]:
from mich.models.blocks import SpatialEncoder

In [ ]:
spatial_enc = SpatialEncoder(
    [
        dict(cin=4, cout=6, stride=1, dw_kernel=3, pw_kernel=1, num_groups=1, activation="silu"),
    ]
)
encoded = spatial_enc(mixed)
print("input :", mixed.shape, " -> output:", encoded.shape, " ([B, T, L, C', H, W])")

### 1c. `TemporalMixingEncoder`: dilated TCN layers

Each layer's dilation is **hard-coded** as `2**i` inside `TemporalMixingEncoder.__init__` (any `dilation`
key you pass in `module_config` is silently overwritten). With the 6-layer config used everywhere in this
repo, dilations are always `1, 2, 4, 8, 16, 32` regardless of YAML.

In [ ]:
from mich.models.blocks import TemporalMixingEncoder

In [ ]:
temporal_enc = TemporalMixingEncoder(
    [
        dict(cin=6, kernel_size=3, num_groups=1, activation="silu"),
        dict(cin=6, kernel_size=3, num_groups=1, activation="silu"),
    ]
)
print("dilations actually used:", [m.depthwise.dilation[0] for m in temporal_enc.module])

temporal_out = temporal_enc(encoded)
print(
    "input :", encoded.shape, " -> output:", temporal_out.shape, " ([B, T, L, C', H, W], unchanged)"
)

### 1d. Time embedding + FiLM: how the decoder becomes time/layer/signal-aware

In [ ]:
from mich.models.blocks import FourierTimeEmbedding

time_embed = FourierTimeEmbedding(num_freqs=8, min_freq=1.0, max_freq=20.0)
t_grid = torch.linspace(0.0, 1.0, 100)
emb = time_embed(t_grid)  # [T, 2*num_freqs]
print("Fourier time embedding shape:", emb.shape, "(sin/cos pairs per frequency)")

fig, ax = plt.subplots(figsize=(8, 3))
for k in range(0, 8, 2):
    ax.plot(t_grid, emb[:, k], label=f"sin, freq idx {k // 2}")
ax.set_xlabel("normalised time")
ax.set_title("A few Fourier time-embedding channels (frequencies log-spaced 1-20 Hz)")
ax.legend(fontsize=10, frameon=False, loc="upper right", bbox_to_anchor=(1.25, 1.0))
plt.show()

### 1e. `SpatioTemporalDecoder`: FiLM-conditioned, per-(signal, layer) output heads

This is the piece that turns encoded spatio-temporal features into the 7 Heinzle states. Output channel
order follows `HEINZLE_SIGNALS = ["x", "s", "f", "v", "q", "vstar", "qstar"]`. Note that the code spells the
drainage variables `vstar`/`qstar` (no asterisk, since it's a Python identifier), while the forward
simulator in `balloon.py` calls the same quantities `v*`/`q*`. Same physical variables, different string
keys in different files.

In [ ]:
from mich.models.blocks import HEINZLE_SIGNALS, SpatioTemporalDecoder

decoder = SpatioTemporalDecoder(
    cin=6,
    c_dec=8,
    out_channels=7,
    activation="silu",
    L=3,
    temporal_film_config=dict(embed_dim=16, hidden_dim=8, activation="silu", c_dec=4),
    temporal_embedding_config=dict(num_freqs=8, min_freq=1.0, max_freq=20.0),
    c_film=4,
)
t_batch = torch.linspace(0.0, 1.0, T).unsqueeze(0).expand(2, -1)  # [B=2, T]
manifest = decoder(temporal_out, t_batch, return_gradients=True)

print("z_hat shape [B, 7, L, T, H, W]:", manifest.z_hat.shape)
print("dz_hat_dt shape (same):", manifest.dz_hat_dt.shape)
print("HEINZLE_SIGNALS order:", HEINZLE_SIGNALS)
print("manifest.channel('x') shape [B, L, T, H, W]:", manifest.channel("x").shape)

## 2. Assembling the real thing: `HeinzleNet` from `config/model/default.yaml`

`HeinzleNet.__init__` just wires the blocks above together in sequence: `MaskedLayerMixing`,
`SpatialEncoder`, `TemporalMixingEncoder`, `SpatioTemporalDecoder`. Rather than re-deriving the config by
hand, we load the actual `config/model/default.yaml` and let Hydra's `instantiate` build it. This is
exactly what `scripts/train_mich.py` does (via `hydra.utils.instantiate(cfg.model)` for the whole `MICH`
module; here we just target the `heinzle_net` sub-block directly).

In [ ]:
import hydra

# `config/model/default.yaml` uses `${model.L}`-style interpolations, which only resolve if the
# file is loaded as the "model" key of some parent config -- exactly how mainconfig.yaml composes it.
raw_model_cfg = OmegaConf.create({"model": OmegaConf.load("../config/model/default.yaml")})
resolved = OmegaConf.to_container(raw_model_cfg, resolve=True)
print(json.dumps(resolved["model"]["heinzle_net"]["layer_mixing_config"], indent=2))
print(json.dumps(resolved["model"]["heinzle_net"]["spatial_decoder_config"], indent=2))

In [ ]:
net = hydra.utils.instantiate(raw_model_cfg.model.heinzle_net)
n_params = sum(p.numel() for p in net.parameters())
print(f"HeinzleNet from config/model/default.yaml: {n_params:,} parameters")
print(net)

### Config key to constructor argument, at a glance

| `config/model/default.yaml` key | Ends up as |
|---|---|
| `L`, `C` | `heinzle_net.layer_mixing_config`, passed to `MaskedLayerMixing(L, C, init_identity)` |
| `spatial_encoder_config` (list of 2 dicts) | `SpatialEncoder(module_config)`, one `DepthWiseSeparableConvLayer(**dict)` per entry |
| `temporal_mixing_config` (list of 6 dicts) | `TemporalMixingEncoder(module_config)`, one `TemporalDepthWiseTCNLayer(**dict, dilation=2**i)` per entry (dilation forced, see §1c) |
| `num_freqs`, `min_freq`, `max_freq` | `time_embedding_config`, passed to `FourierTimeEmbedding(**...)` |
| `time_film_config.{embed_dim,c_dec}` | **overwritten** inside `SpatioTemporalDecoder.__init__` (comment in the YAML says so); real values are derived from `2*num_freqs + layer_embed_dim + signal_embed_dim` and `c_film` |
| `spatial_decoder_config` (`cin, c_dec, c_film, signal_embed_dim, out_channels, L, layer_embed_dim`) | `SpatioTemporalDecoder(**dict, ...)` |
| `normaliser.*` | separate top-level `LayerwiseBOLDNormalizer` (§3 below), not part of `HeinzleNet` itself |
| `optimizer`, `scheduler` (`_partial_: true`) | become `functools.partial`, applied later by `MICH.configure_optimizers` |
| `loss_config`, `learnable_physio` | passed straight through to `MICH.__init__` as plain mappings, see `lightning_framework.md` |

`longfreq.yaml` is the same shape with `L: 1` (single layer, e.g. for single-layer scenarios), a narrower
time-embedding band, and only 5 output signals (`x, s, f, v, q`, no drainage terms). `supervised.yaml`
targets a completely different, simpler network (`FullySupervisedNet`, §4 below).

## 3. Normalising BOLD before it reaches the network

In the real pipeline, BOLD is never fed to `HeinzleNet` raw. It first passes through
`LayerwiseBOLDNormalizer` (`src/mich/models/normaliser.py`), a Welford online normaliser that tracks
running mean/variance from a spatial neighbourhood around the known source voxel (see `training.md` §2 for
the full rationale). We instantiate it the same way `config/model/default.yaml`'s `normaliser` block does.

In [ ]:
from mich.models.normaliser import LayerwiseBOLDNormalizer

normaliser = LayerwiseBOLDNormalizer(
    H=H, W=W, eps=1e-6, freeze_after_steps=320, neighbourhood_radius=2
)

src_pos_t = torch.from_numpy(
    source_position[:, :1, :].astype(np.int64)
)  # (N, S=1, 2), one source slot per sample
num_sources_t = torch.ones(N, dtype=torch.int64)

normaliser.train()
bold_norm = normaliser(bold_t, source_position=src_pos_t, num_sources=num_sources_t)
print("raw BOLD range:      [%.4f, %.4f]" % (bold_t.min(), bold_t.max()))
print("normalised BOLD range: [%.4f, %.4f]" % (bold_norm.min(), bold_norm.max()))

## 4. A forward pass on real data

The network below is **randomly initialised** (we haven't trained it), so its predictions are not
expected to match the simulator's ground truth. The point here is purely to exercise the API: input
shape, output shape, the `SpatialDecoderManifest` accessors, and the analytic time-derivative path.

In [ ]:
t_batch = torch.linspace(0.0, 1.0, T).unsqueeze(0).expand(N, -1)  # [B, T]

net.eval()
with torch.no_grad():
    manifest = net(bold_norm, t_batch, return_gradients=False)

print("z_hat:", manifest.z_hat.shape, "  (B, 7, L, T, H, W)")
for signal in ["x", "s", "f", "v", "q", "vstar", "qstar"]:
    chan = manifest.channel(signal)
    print(f"  {signal:>6}: range [{chan.min():.3f}, {chan.max():.3f}]")

In [ ]:
# Untrained predicted x vs the simulator's ground-truth x, at the source voxel of sample 0.
sample_idx = 0
layer = int(source_layer[sample_idx, 0])
i, j = source_position[sample_idx, 0]

pred_x = manifest.channel("x")[sample_idx, layer, :, i, j].numpy()
true_x = x_true[sample_idx, layer, :, i, j]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(true_x, label="ground-truth x (from simulator)")
ax.plot(pred_x, label="HeinzleNet predicted x (untrained!)", alpha=0.7)
ax.set_xlabel("timestep")
ax.legend(frameon=False, fontsize=10, loc="upper right")
ax.set_ylim(-0.1, 1.4)
ax.set_title("Untrained model: shapes/API check, not an accuracy check")
plt.show()

In [ ]:
# Now with analytic time derivatives (needed for the physics loss -- see lightning_framework.md).
with torch.no_grad():
    manifest_grad = net(bold_norm, t_batch, return_gradients=True)
print("dz_hat_dt:", manifest_grad.dz_hat_dt.shape)

ds_dt = manifest_grad.channel_grad("s")[sample_idx, layer, :, i, j].numpy()
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ds_dt)
ax.set_title("Analytic d(s_hat)/dt at the source voxel (untrained network)")
ax.set_xlabel("timestep")
plt.show()

## 5. `MICH`: the LightningModule that wraps `HeinzleNet`

`HeinzleNet` itself has no notion of training, losses, or optimisers. That role belongs to `MICH`
(`src/mich/models/mich.py`), a `LightningModule` that contains a `HeinzleNet` instance as
`self.heinzle_net` (it is not a subclass of it). `MICH.forward` is a thin wrapper: normalise, then call
`self.heinzle_net(...)`.

`scripts/train_mich.py` injects a few physiological hparams (`haemo`, `acquisition`, `V0`) into
`cfg.model` at runtime, reading them from whichever simulation HDF5 the datamodule points at, so the
model's constants always match the data it's trained on (see `lightning_framework.md`). Constructing
`MICH` directly (outside the full training script) means supplying those yourself. `tests/test_mich.py`
shows the minimal pattern:

In [ ]:
import types
from functools import partial

from mich.models.mich import MICH

mich_model = MICH(
    heinzle_net=net,
    normaliser=normaliser,
    optimizer=partial(torch.optim.AdamW, lr=3e-4, weight_decay=1e-5),
    scheduler=partial(torch.optim.lr_scheduler.CosineAnnealingLR, T_max=100, eta_min=1e-6),
    loss_config=resolved["model"]["loss_config"],
    haemo=types.SimpleNamespace(
        kappa=1.92, gamma=0.41, alpha=0.32, tau=2.66, lambda_d=1.0, tau_d=1.0
    ),
    acquisition=types.SimpleNamespace(
        k1=4.3 * 28.625 * 0.34 * 0.25, k2=1e-32 * 340 * 0.34 * 0.25, k3=1 - 1e-32, E0=0.34
    ),
    V0=0.05,
    lightning={"interval": "step", "frequency": 1},
)
mich_model.eval()
with torch.no_grad():
    mich_manifest = mich_model(bold_t, t_batch, normalise=True)
print("MICH.forward output z_hat:", mich_manifest.z_hat.shape)

## 6. Where to go next

- **`lightning_framework.md`**: how `MICH` fits into the full Hydra + PyTorch Lightning training loop,
  what `scripts/train_mich.py` actually does with the model above, which config files to edit for common
  changes, and how to launch/monitor/resume a training run.
- **`training.md`**: the loss design in depth (data loss, physics loss, the collocation sampler, loss
  scheduling).
- **`heinzlenet.md`**: the conceptual rationale behind each architectural choice (keep the two caveats from
  the top of this notebook in mind when reading it).
- Try re-running §2-4 with `config/model/longfreq.yaml` or `config/model/supervised.yaml` instead of
  `default.yaml` to see the single-layer and fully-supervised variants.